# Word Embeddings

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/04_word_embeddings.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️

## Setup

### Housekeeping (no interaction required)

In [2]:
%pip install gensim

/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


❗ Please restart the kernel/runtime after installing the package to ensure that all changes take effect.kl

(Google Colab might initiate a restart on its own)

In [3]:
import os
from pathlib import Path
from typing import Any, Dict, Iterator, Literal

# from IPython.display import clear_output
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

In [4]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [5]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Ask for a yes/no confirmation before running a step.

    Args:
        question: Prompt shown to the user.

    Returns:
        True if the answer is yes, otherwise False.
    """
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

### Setup (interaction required)

In [ ]:
### ⬇️⬇️⬇️ Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False # Set to True if you want to use your own data
### ⬆️⬆️⬆️

<img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=12> The cell below will attempt to connect to your Google Drive.

*Once prompted, give all demanded permissions*

In [7]:
if IN_COLAB and USE_YOUR_DATA: #confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data

### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [10]:
if USE_YOUR_DATA:
    print("Loading your raw data...", end=" ")
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)
    print("Done!")

    print("Loading your lemma data..", end=" ")
    LEMMA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet"
    lemma_df = pd.read_parquet(LEMMA_PATH)
    print("Done!")

    print("Loading your token data..", end=" ")
    TOKENS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet"
    wordform_df = pd.read_parquet(TOKENS_PATH)
    print("Done!")

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

Loading your raw data... Done!
Loading your lemma data.. Done!
Loading your token data.. Done!
Data loaded successfully! 2633 rows.


### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [ ]:
if not USE_YOUR_DATA:
    RAWDATA_URL = f"https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/{CORPUS_NAME}.filtered.parquet"
    print(f"Loading raw data ...", end=" ")
    raw_df = pd.read_parquet(RAWDATA_URL)
    print("Done!")

    LEMMA_URL = f"https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/{CORPUS_NAME}.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end=" ")
    lemma_df = pd.read_parquet(LEMMA_URL)
    print("Done!")

    TOKENS_URL = f"https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/{CORPUS_NAME}.filtered.tokens.parquet"
    print(f"Loading token data ...", end=" ")
    wordform_df = pd.read_parquet(TOKENS_URL)
    print("Done!")

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

### Parsing

The library that calculates the Word2Vec model has specific demands for the datatype of the texts.

Loading from parquet format creates `np.ndarray` objects for the sentences and tokens, but the code needs objects of type `list`.

In [17]:
import numpy as np

def convert_iterable_to_list(obj: Any) -> Any:
    """Recursively convert nested numpy arrays to plain Python lists.

    Args:
        obj: Any nested object that may contain numpy arrays.

    Returns:
        The same structure, but with numpy arrays replaced by lists.
    """
    if isinstance(obj, np.ndarray):
        return [convert_iterable_to_list(item) for item in obj]
    return obj

raw_df['tokens'] = raw_df['tokens'].progress_apply(convert_iterable_to_list)
raw_df['lemmas'] = raw_df['lemmas'].progress_apply(convert_iterable_to_list)


100%|██████████| 2633/2633 [00:00<00:00, 635676.19it/s]

100%|██████████| 2633/2633 [00:00<00:00, 857422.55it/s]


## Word2Vec

#### Calculate Model

In [18]:
from gensim.models import Word2Vec

# Memory-efficient way to iterate over tokenized sentences
class SentenceIterator:
    """Yield sentence token lists from a dataframe column.

    This avoids building one giant list in memory.
    """

    def __init__(self, df: pd.DataFrame, column: Literal["tokens", "lemmas"]):
        """Store dataframe and target column.

        Args:
            df: Input dataframe containing tokenized documents.
            column: Column with nested sentence token lists.
        """
        self.df = df
        self.column = column

    def __iter__(self) -> Iterator[list[str]]:
        """Iterate over all sentences in all documents."""
        for document in tqdm(self.df[self.column], leave=False, desc="Iterating over sentences"):
            for sentence in document:
                yield sentence

The cell below trains the Word Embeddings on `lemmas` as opposed to `tokens`.

Lemmas have a smaller Type/Token-Ratio (TTR), which benefits the Word2Vec-model especially when being created for small corpora.

$\frac{\#\text{lemma}_{unique}}{\#\text{lemma}} \ll \frac{\#\text{token}_{unique}}{\#\text{token}}$

For example, the vector for the lemma *'Haus'* will be informed by all tokens that have that lemma, e.g. *'Häuser'* (plural) or *'Hauses'* (genitive).
In the other case, the vector for token *'Hauses'* would only be informed by the sentences that contain exactly this word.

![Type-Token-Ratio for Wordform and Lemmas](../assets/ttr_we.excalidraw.png)

In [ ]:
sentences = SentenceIterator(raw_df, "lemmas")
model = Word2Vec(sentences=sentences, vector_size=300, window=5, min_count=5, workers=4)

#### Extract and save vectors

In [20]:
if USE_YOUR_DATA:
    wv = model.wv
    wv.save_word2vec_format(DATA_DIR / f"{CORPUS_NAME}.word_vectors", binary=True)

## Load calculated vectors

In [21]:
from gensim.models import KeyedVectors
import urllib.request

if USE_YOUR_DATA:
    wv = KeyedVectors.load_word2vec_format(str(DATA_DIR / f"{CORPUS_NAME}.word_vectors"), binary=True)
else:
    MODEL_URL = f"https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/{CORPUS_NAME}.word_vectors"
    urllib.request.urlretrieve(MODEL_URL, "/tmp/model.word_vectors")
    wv = KeyedVectors.load_word2vec_format("/tmp/model.word_vectors", binary=True)

#### Most similar words

In [ ]:
# ⬇️⬇️⬇️ Adjust here to test the model with your own words
SEARCH_TERM = "Geld"
# ⬆️⬆️⬆️

wv.most_similar_cosmul(SEARCH_TERM)

In [ ]:
# ⬇️⬇️⬇️ 
START_TERM = "Armut"
MINUS_TERM = "verschuldet"
PLUS_TERM = "unverschuldet"
# ⬆️⬆️⬆️

wv.most_similar_cosmul(positive=[START_TERM, PLUS_TERM], negative=[MINUS_TERM])[:10]

In [ ]:
# ⬇️⬇️⬇️ 
START_TERM = "Armensteuer"
MINUS_TERM = "Staat"
PLUS_TERM = "Familie"
# ⬆️⬆️⬆️

wv.most_similar_cosmul(positive=[START_TERM, PLUS_TERM], negative=[MINUS_TERM])[:10]

In [22]:
import csv
from os import PathLike
from gensim.models import KeyedVectors

def wvec2tsv(
    infile: str | PathLike[str],
    outf_vecs: str | PathLike[str],
    outf_meta: str | PathLike[str],
    binary: bool = True,
) -> None:
    """Convert word2vec vectors into TSV files for TensorFlow Projector.

    Args:
        infile: Path to the word2vec file.
        outf_vecs: Output path for vectors TSV.
        outf_meta: Output path for metadata TSV.
        binary: Whether the input word2vec file is binary.
    """
    wv_local = KeyedVectors.load_word2vec_format(str(infile), binary=binary)

    with open(outf_vecs, "w", encoding="utf-8", newline="") as vec_tsv:
        with open(outf_meta, "w", encoding="utf-8", newline="") as meta_tsv:
            vec_writer = csv.writer(vec_tsv, delimiter="\t")
            meta_writer = csv.writer(meta_tsv, delimiter="\t")

            for token in tqdm(wv_local.index_to_key, desc="Converting word vectors to TSV format"):
                meta_writer.writerow([token])
                vec_writer.writerow(wv_local[token].tolist())

wvec2tsv(
    DATA_DIR / f"{CORPUS_NAME}.word_vectors",
    DATA_DIR / f"{CORPUS_NAME}.vecs.tsv",
    DATA_DIR / f"{CORPUS_NAME}.meta.tsv",
    binary=True,
)

if IN_COLAB:
    from google.colab import files
    files.download(str(DATA_DIR / f"{CORPUS_NAME}.vecs.tsv"))
    files.download(str(DATA_DIR / f"{CORPUS_NAME}.meta.tsv"))










































































































Converting word vectors to TSV format: 100%|██████████| 46304/46304 [00:10<00:00, 4252.44it/s]
